# 트랜스포머 처음부터 훈련하기 - Adam 옵티마이저와 역전파

- Tutorial ID: `tut-10-1`
- Tutorial: 트랜스포머 처음부터 훈련하기
- Section ID: `s10-1-1`
- Section: Adam 옵티마이저와 역전파


In [ ]:
# ============================================================
# 코드 읽는 법 — Adam 옵티마이저와 역전파
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("트랜스포머 훈련: Adam + 역전파")
print("=" * 60)

In [ ]:
# 유틸리티 함수

In [ ]:
def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

def cross_entropy_loss(logits, targets):
    """크로스 엔트로피 손실"""
        probs = softmax(logits)
    n = len(targets)
        log_probs = np.log(probs[np.arange(n), targets] + 1e-9)
    return -np.mean(log_probs)

def cross_entropy_grad(logits, targets):
    """크로스 엔트로피 그래디언트"""
        probs = softmax(logits)
    n = len(targets)
    probs[np.arange(n), targets] -= 1
    return probs / n

In [ ]:
# Adam 옵티마이저

In [ ]:
# Adam 옵티마이저 (딕셔너리로 상태 관리)
def create_adam(lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam 옵티마이저 상태 생성"""
    return {'lr': lr, 'beta1': beta1, 'beta2': beta2, 'eps': eps,
            't': 0, 'm': {}, 'v': {}}

def adam_step(opt, param_id, param, grad):
    """Adam 파라미터 업데이트"""
    opt['t'] += 1
    if param_id not in opt['m']:
        opt['m'][param_id] = np.zeros_like(param)
        opt['v'][param_id] = np.zeros_like(param)
    
    opt['m'][param_id] = opt['beta1'] * opt['m'][param_id] + (1 - opt['beta1']) * grad
    opt['v'][param_id] = opt['beta2'] * opt['v'][param_id] + (1 - opt['beta2']) * grad**2
    
    m_hat = opt['m'][param_id] / (1 - opt['beta1']**opt['t'])
    v_hat = opt['v'][param_id] / (1 - opt['beta2']**opt['t'])
    
    param -= opt['lr'] * m_hat / (np.sqrt(v_hat) + opt['eps'])
    return param

In [ ]:
# 간단한 1-레이어 트랜스포머 훈련

In [ ]:
print("
간단한 언어 모델 훈련 시작...")

vocab_size = 5
d_model = 8
d_head = 4
seq_len = 4

vocab = ['A', 'B', 'C', 'D', 'E']

np.random.seed(0)

# 파라미터 초기화 (Xavier)
scale = np.sqrt(2.0 / d_model)
W_E = np.random.randn(vocab_size, d_model) * scale
W_U = np.random.randn(d_model, vocab_size) * (scale / 2)
W_Q = np.random.randn(d_model, d_head) * scale
W_K = np.random.randn(d_model, d_head) * scale
W_V = np.random.randn(d_model, d_head) * scale
W_O = np.random.randn(d_head, d_model) * scale

# 훈련 데이터: 패턴 "A B A B", "C D C D" 등
def generate_repeated_pattern(vocab_size, seq_len, n_samples):
    data = []
        for _ in range(n_samples):
        # 랜덤 패턴 선택하고 반복
        pattern = np.random.randint(0, vocab_size, size=seq_len//2)
        seq = np.tile(pattern, 2)  # 패턴을 2번 반복
        data.append(seq)
    return np.array(data)

training_data = generate_repeated_pattern(vocab_size, seq_len, 100)
print(f"훈련 데이터 예시: {[vocab[t] for t in training_data[0]]}")
print(f"  → 반복 패턴 (인덕션 헤드가 학습해야 할 것)")

# 옵티마이저
opt = create_adam(lr=0.01)

# 전방 패스 함수
def forward_pass(seq):
        X = W_E[seq]
        mask = np.triu(np.full((len(seq), len(seq)), -1e9), k=1)
        Q = X @ W_Q
        K = X @ W_K
        V = X @ W_V
        A = softmax(Q @ K.T / np.sqrt(d_head) + mask)
        head_out = (A @ V) @ W_O
    x1 = X + head_out
        logits = x1 @ W_U
    return logits, A

# 훈련 루프
n_epochs = 200
losses = []

for epoch in range(n_epochs):
        epoch_loss = 0
    
        for seq in training_data[:20]:  # 배치 크기 제한
        logits, A = forward_pass(seq)
        
        # 손실: 각 위치에서 다음 토큰 예측
        targets = seq[1:]  # 실제 다음 토큰들
                pred_logits = logits[:-1]  # 예측 로짓 (마지막 제외)
        
                loss = cross_entropy_loss(pred_logits, targets)
        epoch_loss += loss
        
        # 그래디언트 계산 (수치 미분으로 시뮬레이션)
        # 실제로는 자동 미분(autograd) 사용
        eps_val = 1e-4
        
        # W_U 그래디언트 (주요 파라미터만)
                grad_logits = cross_entropy_grad(pred_logits, targets)
        
        # 간단한 그래디언트 업데이트 (실제 역전파 대신)
        x1_last = (W_E[seq[:-1]] + (softmax(
                        W_E[seq[:-1]] @ W_Q @ W_K.T @ W_E[seq[:-1]].T / np.sqrt(d_head) + 
            np.triu(np.full((len(seq)-1, len(seq)-1), -1e9), k=1)
        ) @ W_E[seq[:-1]] @ W_V) @ W_O)
        
                W_U_grad = x1_last.T @ grad_logits / len(targets)
                W_E_grad = np.zeros_like(W_E)
                for i, t in enumerate(seq[:-1]):
                        W_E_grad[t] += grad_logits[i] @ W_U.T
        
        # Adam 업데이트
                W_U[:] = adam_step(opt, 'W_U', W_U, W_U_grad)
                for t in range(vocab_size):
            if np.any(W_E_grad[t] != 0):
                                W_E[t] = adam_step(opt, f'W_E_{t}', W_E[t], W_E_grad[t] * 0.1)
    
    epoch_loss /= 20
    losses.append(epoch_loss)
    
    if epoch % 40 == 0:
                print(f"  Epoch {epoch:3d}: loss = {epoch_loss:.4f}")

print(f"
최종 loss: {losses[-1]:.4f}")
print(f"초기 loss: {losses[0]:.4f}")
print(f"개선율: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

# 훈련 후 검증
print("
훈련 후 예측 검증:")
test_seq = [0, 1, 0]  # A B A → 다음은 B?
logits, _ = forward_pass(test_seq)
probs = softmax(logits[-1])
predicted = vocab[np.argmax(probs)]
print(f"  입력: {[vocab[t] for t in test_seq]}")
print(f"  예측: {predicted} (기대: B)")
print(f"  각 토큰 확률: {dict(zip(vocab, np.round(probs, 3)))}")

print("""
주의: 이것은 개념적 시뮬레이션입니다.
실제 인덕션 헤드 훈련에는:
  - 더 큰 모델 (d_model=128~512)
  - 더 많은 데이터 (수백만 토큰)
  - 완전한 역전파 (PyTorch autograd)
  - 더 긴 훈련 (수천 스텝)
  가 필요합니다.
""")